In [130]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [131]:
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [132]:
data.drop(['RowNumber', 'CustomerId', 'Surname'], axis= 1, inplace=True)

In [133]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [134]:
data['Geography'].value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [135]:
data['Gender'].value_counts()

Gender
Male      5457
Female    4543
Name: count, dtype: int64

In [136]:
lb = LabelEncoder()

data['Gender'] = lb.fit_transform(data['Gender'])
# data['Geography'] = lb.fit_transform(data["Geography"]) ## Bad -> the geography with value 2 might be enterpreted as greater by model

In [137]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [138]:
## OneHot encoding for Geo
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder()
a = ohe.fit_transform(data[["Geography"]])
b = ohe.get_feature_names_out(["Geography"])

In [139]:
a.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [140]:
a,b

(<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 10000 stored elements and shape (10000, 3)>,
 array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
       dtype=object))

In [141]:
df = pd.DataFrame(data= a.toarray(),columns=["Geo_France", "Geo_Germany", "Geo_Spain"])

In [142]:
df

,Geo_France,Geo_Germany,Geo_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [143]:
## Combine

data.drop("Geography", axis = 1, inplace= True)
data = pd.concat([data, df], axis= 1)

In [144]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geo_France,Geo_Germany,Geo_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [145]:
## Save Encoders

with open("label_encoder_gender.pkl", "wb") as file:
    pickle.dump(lb, file)

In [ ]:
with open("onehot_encoder_geo.pkl", "wb") as file:
    pickle.dump(ohe, file)

In [147]:
## Dependent and Independent features
X = data.drop("Exited", axis = 1)
y = data["Exited"]


In [148]:
## Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size= 0.2, random_state=42)

## Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [149]:
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

#### ANN implementation

In [150]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [151]:
print((X_train.shape[1],))
print(X_train.shape[1])

(12,)
12


In [152]:
## Build ANN Model
model = Sequential(
    [
        Dense(64, activation="relu", input_shape = (X_train.shape[1],)), ## HL1 (connected with input layer) with 64 neurons (Dense(64)), ReLU activation function, how many inputs are there (input_shape)
        Dense(32, activation="relu"), ## HL2 # No need of specifying input_shape
        Dense(1, activation="sigmoid") ## O/P layer
    ]
)

# Model done

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [153]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01) ## Flexible method for optimizer initiation (can set your own LR)
LOSS = tf.keras.losses.BinaryCrossentropy() ## another way

In [154]:
## compile model for forward and backward propagation

# model.compile(optimizer="adam", loss="binary_crossentropy", metrics=['accuracy']) ## Non flexible way of initating optimizer (LR fixed)
model.compile(optimizer=opt, loss=LOSS, metrics=['accuracy']) 

In [155]:
## Set up the Tensorboard for visulaize all the logs while training model
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S") ## for saving logs while training
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [156]:
## Set up Early Stopping -> If in 100 Epocs training model achieved it very best at 20 and after it only 1%-2% accuracy is getting increased or the loss value is not decreasing more then we should stop at 20

early_stopping_callback = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
# patience=x parameter tells that "wait for atleast x epochs before making any decision"
# monitor="abc" tells what thing has to be monitored for early stopping
# restore_best_weights = True tells that whenever training finishes take the best weight whenever (in any epoch) it came

In [157]:
## Training

history = model.fit(
    X_train, y_train, validation_data = (X_test, y_test) , epochs = 100,
    callbacks = [tensorflow_callback, early_stopping_callback]
)

Epoch 1/100
250/250 [==============================] - 2s 4ms/step - loss: 0.4001 - accuracy: 0.8341 - val_loss: 0.3918 - val_accuracy: 0.8470
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3549 - accuracy: 0.8546 - val_loss: 0.3482 - val_accuracy: 0.8565
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3443 - accuracy: 0.8595 - val_loss: 0.3488 - val_accuracy: 0.8545
Epoch 4/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3415 - accuracy: 0.8591 - val_loss: 0.3425 - val_accuracy: 0.8565
Epoch 5/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3397 - accuracy: 0.8597 - val_loss: 0.3499 - val_accuracy: 0.8590
Epoch 6/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3360 - accuracy: 0.8640 - val_loss: 0.3520 - val_accuracy: 0.8560
Epoch 7/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3362 - accuracy: 0.8625 - val_loss: 0.3531 - val_accuracy: 0.8550

In [158]:
## save model

model.save("model.h5") ## .h5 is compatible with keras

d:\AIML Material\venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [159]:
## Load Tensorboard Extension

%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [161]:
%tensorboard --logdir logs/fit/20260609-214755